# 이론을 읽고 계산·실습으로 확인하기

먼저 아래 설명으로 전체 흐름을 이해합니다. 그다음 준비 셀을 실행하고, 배운 개념을 계산으로 확인합니다.

처음에는 코드를 실행하지 않습니다. **무엇을 배우는지 한 번 훑고 → 개념을 이해하고 → 작은 계산과 실습으로 확인**합니다.

## 1. 이 프로젝트는 무엇을 만드는가?

사진을 보고 고양이인지 자동차인지 고르는 계산기를 만듭니다. 먼저 소프트웨어 모델이 어떻게 판단하는지 배웁니다. 그다음 같은 계산을 빠르게 처리할 회로를 만듭니다. 이것이 딥러닝 가속기입니다.

## 2. 사진이 답으로 바뀌는 큰 흐름

**사진 → 숫자 → 특징 찾기 → 클래스별 점수 → 가장 큰 점수 선택**

- 사진: 컴퓨터는 색을 숫자로 읽습니다.
- 특징: 작은 무늬와 그 조합을 계산합니다.
- 점수: 고양이·자동차 등 후보마다 점수를 냅니다.
- 예측: 가장 점수가 높은 후보를 고릅니다.

이 계산 방법을 묶은 것이 **모델**입니다. 계산에 쓰는 조절값이 **Weight와 Bias**입니다.

## 3. 모델은 어떻게 배우는가?

학생이 연습문제를 풀고 답을 맞춰보는 과정에 비유해 봅시다.

| 공부할 때 | 모델에서 |
| --- | --- |
| 문제를 보고 답하기 | 입력으로 예측하기 |
| 정답과 비교해 채점하기 | Loss 계산하기 |
| 어디를 고쳐야 할지 찾기 | Gradient 계산하기 |
| 풀이 방법 고치기 | Weight·Bias 갱신하기 |
| 처음 보는 문제 풀기 | 새 데이터로 평가하기 |

**학습은 계산에 쓰는 값을 고치는 과정, 추론은 배운 값을 써서 답하는 과정**입니다. 우리 회로는 주로 추론 계산을 맡습니다.

## 4. 먼저 알아둘 말

| 말 | 짧게 설명하면 |
| --- | --- |
| Tensor·Shape | 숫자를 담은 배열·배열의 크기 |
| Weight | 입력에 곱하는 숫자 |
| Bias | 곱한 값들을 합친 뒤 더하는 기본값 |
| ReLU | 음수는 0, 양수는 그대로 두는 계산 |
| Loss | 예측을 채점한 벌점. 작을수록 좋음 |
| Gradient | 조절값을 조금 바꿀 때 벌점이 어떻게 달라지는지 알려주는 값 |
| CNN·Conv | 작은 필터로 이미지의 여러 위치를 계산하는 모델·계산 |
| Psum | 곱한 값들을 더해 가는 중간 합 |
| INT8 | 숫자를 8비트 정수로 표현하는 방식 |
| RTL | 회로가 클록마다 무엇을 할지 적은 코드 |

한 번에 외우지 않아도 됩니다. 아래 순서로 읽으며 같은 예제에 붙여 봅니다.

## 5. 개념을 읽는 순서

1. **모델의 기본:** A0 → A1 → A2 → A3. 입력·Weight·Bias·Layer를 이해합니다.
2. **학습과 평가:** A4 → A5. 예측·채점·수정·평가를 구분합니다.
3. **이미지를 계산하는 방법:** B1 → B2 → B3 → A7 → B4 → B5. Conv에서 ResNet-20 전체 흐름으로 이어집니다.
4. **회로로 옮기는 방법:** A6 → D1 → D2 → D3 → D5. 정수·누산·메모리·대기를 이해합니다.

각 장의 앞쪽은 쉬운 개념 설명입니다. 뒤의 수식·세부 규약은 필요할 때 펼칩니다. D4·D6~D8과 논문은 이후 심화 자료입니다.

## 6. 이제 계산과 실습으로 확인합니다

‘Weight는 입력에 곱하는 값’, ‘Conv는 작은 필터의 곱·합’, ‘학습과 추론은 다르다’를 자기 말로 설명할 수 있으면 S0로 갑니다.

실습 순서는 S0 작은 Conv → S1 실제 이미지 → S2 학습·Residual → S3 정수 → S4 RTL → S6 보드입니다. S5 실제 모델 연결은 선택입니다. **실습은 앞에서 배운 개념을 확인하는 단계**입니다.


[차례대로 개념 읽기](https://github.com/tsi03048/resnet20-accel/blob/main/docs/learning/README.md)

---

이제 계산과 실습으로 확인합니다.

In [ ]:
# 0. 저장소 준비. 다시 실행해도 폴더 안에 중첩 복제하지 않는다.
import os, sys, subprocess
from pathlib import Path
repo = Path('/content/resnet20-accel') if Path('/content').is_dir() else (Path.cwd() if (Path.cwd() / 'model/common.py').exists() else Path.cwd() / 'resnet20-accel')
if not (repo / 'model/common.py').exists():
    subprocess.run(['git', 'clone', 'https://github.com/tsi03048/resnet20-accel.git', str(repo)], check=True)
os.chdir(repo)
sys.path.insert(0, str(repo))
print('실행 폴더:', repo)
subprocess.run(['git', 'rev-parse', 'HEAD'], check=True)


## S0 먼저 예측하기

4×4 입력에 3×3 필터를 적용하면 왜 출력이 네 개일까요? X[0][0]만 5→6으로 바꾸면 어느 출력이 변할까요?

In [ ]:
subprocess.run([sys.executable, '-m', 'exercises.conv_walkthrough'], check=True)
subprocess.run([sys.executable, '-m', 'exercises.step_up', 'conv'], check=True)

## S1 환경 준비

Colab은 기본 PyTorch를 사용합니다. 다음 단계는 한 장만 실행합니다.

In [ ]:
import importlib.util
for package in ('numpy', 'PIL', 'torch', 'torchvision'):
    if importlib.util.find_spec(package) is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)

## 이미지 한 장 예측

입력 [1,3,32,32]는 RGB 이미지 한 장입니다. 출력 [1,10]은 클래스별 점수입니다. 정답을 보지 않고 가장 큰 점수만 골라 보세요.

In [ ]:
subprocess.run([sys.executable, '-m', 'model.first_prediction', '--device', 'auto'], check=True)
import json
from IPython.display import display, Image, SVG
pred = json.loads(Path('results/learning/first_prediction/prediction.json').read_text(encoding='utf-8'))
display(Image(filename='results/learning/first_prediction/image.png'))
print('정답:', pred['label'], '| 예측:', pred['prediction'], '| logits:', pred['logits_shape'])

In [ ]:
# Logit 0을 기준으로 양수·음수 점수를 같은 눈금에 표시합니다.
scores = pred['logits']
scale = 240 / max(max(abs(v) for v in scores), 1e-9)
parts = ['<svg xmlns="http://www.w3.org/2000/svg" width="820" height="410"><rect width="100%" height="100%" fill="white"/><line x1="390" x2="390" y1="15" y2="400" stroke="#899"/>']
for i, (name, v) in enumerate(zip(pred['classes'], scores)):
    y = 18 + i*38
    x = 390 + min(v, 0)*scale
    width = abs(v)*scale
    color = '#087f80' if i == pred['prediction_index'] else '#7294c8'
    parts.append(f'<text x="8" y="{y+20}" font-size="18">{name}</text><rect x="{x}" y="{y}" width="{width}" height="25" fill="{color}"/><text x="660" y="{y+20}" font-size="18">{v:.3f}</text>')
parts.append('</svg>')
display(SVG(''.join(parts)))

## 값을 바꿔 예측하기

점수를 복사해 다른 클래스의 값을 최댓값보다 1 크게 만들면 무엇이 선택될까요? 이것은 Argmax 규칙 실험이며 모델 학습 결과가 아닙니다.

In [ ]:
changed_scores = scores.copy()
other = (pred['prediction_index'] + 1) % 10
changed_scores[other] = max(scores) + 1
print('선택:', pred['classes'][max(range(10), key=lambda i: changed_scores[i])])

## S2 짧은 학습과 Residual 실험

먼저 w=0.5, Gradient=-2, 학습률=0.1의 새 Weight를 계산하세요. 이어 Conv2 뒤에 ReLU를 추가하면 어떤 값이 사라지는지 예측합니다.

In [ ]:
for lesson in ('learn', 'residual'):
    subprocess.run([sys.executable, '-m', 'exercises.step_up', lesson, '--out', f'results/learning/{lesson}.json'], check=True)

## S3 정수 단위 실험

code 8·Scale 1/8과 code 4·Scale 1/4를 더해 출력 Scale 1/8로 표현하면 얼마일까요?

In [ ]:
subprocess.run([sys.executable, '-m', 'exercises.step_up', 'quant'], check=True)

## 선택: FP32 전체 평가

한 장의 정답을 전체 정확도로 기록하지 않습니다. 아래 True를 선택하면 Test 10,000장을 내려받아 평가하므로 추가 시간이 듭니다.

In [ ]:
RUN_FULL_FP32 = False
if RUN_FULL_FP32:
    if importlib.util.find_spec('datasets') is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', 'datasets'], check=True)
    subprocess.run([sys.executable, '-m', 'model.run_reference', '--device', 'auto', '--e1-only'], check=True)
else:
    print('이번 실행은 한 장 예측입니다. 저장소 기존 전체 평가 JSON은 본인 실행 결과가 아닙니다.')

## 선택: E2-A PTQ 후보 실험

보정 Train 1,000장과 겹치지 않는 개발 Train 1,000장으로 설정을 고른 뒤 한 설정만 Test에 평가합니다. Bias는 FP32입니다. 실제 정수 기준 E2-B는 별도 구현·검증 완료 상태이며, 다음 노트북에서 한 장을 재현합니다. 허용 하락 폭은 팀이 정하고 None이면 자동 승인하지 않습니다.

In [ ]:
RUN_PTQ = False
ACCEPT_DROP_PP = None
if RUN_PTQ:
    if importlib.util.find_spec('datasets') is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', 'datasets'], check=True)
    command = [sys.executable, '-m', 'model.run_reference', '--device', 'auto']
    if ACCEPT_DROP_PP is not None:
        command += ['--accept-drop-pp', str(ACCEPT_DROP_PP)]
    subprocess.run(command, check=True)
else:
    print('PTQ는 실행하지 않았습니다.')

## 본인 결과 저장

배포된 기존 전체 평가 결과와 섞이지 않도록 첫 예측·짧은 실습 폴더만 저장합니다. 전체 평가를 실행했다면 results/run_manifest.json과 해당 결과도 별도로 보관하세요.
[E2-B 정수 한 장](https://colab.research.google.com/github/tsi03048/resnet20-accel/blob/main/notebooks/E2B_integer_reference.ipynb)

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
with ZipFile('learning_outputs.zip', 'w', ZIP_DEFLATED) as z:
    for p in Path('results/learning').rglob('*'):
        if p.is_file(): z.write(p)
try:
    from google.colab import files
    files.download('learning_outputs.zip')
except ImportError:
    print(Path('learning_outputs.zip').resolve())